# CrisisLens: HumSet audit and baselines (v0.1)

Companion notebook for [MHKhan50/CrisisLens-AgentBench](https://github.com/MHKhan50/CrisisLens-AgentBench).
HumSet (Fekih et al., Findings of EMNLP 2022; Apache-2.0) is downloaded from the Hugging Face Hub at run time and not redistributed.

Steps: provenance manifest, dataset audit (languages, labels, duplicates, document leakage across splits), then a prior baseline and a TF-IDF + logistic-regression baseline on all five label sets, on the official split and on a document-grouped split, with document-level bootstrap intervals and per-language scores.

Kaggle settings: **CPU**, **Internet on**. *Save Version → Save & Run All*. About 1–1.5 hours. Outputs: `/kaggle/working/crisislens_v01_reports.zip`.

Humanitarian research only: not for targeting, surveillance, person-level profiling or automated aid decisions.

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("CrisisLens-AgentBench"):
    subprocess.run(["git", "clone", "-q", "https://github.com/MHKhan50/CrisisLens-AgentBench.git"], check=True)
os.chdir("CrisisLens-AgentBench")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
DATA, REPORTS = "/kaggle/working/data/humset", "/kaggle/working/reports"
def cl(*args):
    """Run one CLI step, stream its output and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "crisislens_agentbench.cli", *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True); lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 1. Download with provenance (revision and SHA-256 of every file)

In [ ]:
cl("humset-download", "--data-dir", DATA)

## 2. Smoke test on a small subset

In [ ]:
cl("humset-baselines", "--data-dir", DATA, "--reports", "/kaggle/working/smoke", "--limit", "800", "--n-boot", "50", "--tasks", "sectors")
import shutil; shutil.rmtree("/kaggle/working/smoke", ignore_errors=True)

## 3. Dataset audit

In [ ]:
cl("humset-audit", "--data-dir", DATA, "--reports", REPORTS)

## 4. Baselines: official split and document-grouped split

In [ ]:
# Reuse task results finished in an earlier version, if its output is attached (Add Input).
import glob, os, shutil
os.makedirs(REPORTS + "/runs", exist_ok=True)
for f in glob.glob("/kaggle/input/**/runs/*.json", recursive=True) + glob.glob("/kaggle/input/**/runs/*.npz", recursive=True):
    if not os.path.exists(os.path.join(REPORTS, "runs", os.path.basename(f))):
        shutil.copy(f, REPORTS + "/runs/"); print("restored", os.path.basename(f))
cl("humset-baselines", "--data-dir", DATA, "--reports", REPORTS)
import shutil, json
shutil.copy(DATA + "/manifest.json", REPORTS + "/manifest.json")
shutil.make_archive("/kaggle/working/crisislens_v01_reports", "zip", REPORTS)
print(open(REPORTS + "/audit.md").read()); print(open(REPORTS + "/baselines.md").read())